# `tia` — a video-native backbone: V-JEPA 2 features into the shared cache

The direction's video stream is normally DINOv2: 16 face crops, one CLS vector each. This
notebook builds the alternative — **V-JEPA 2 ViT-L** (Meta, MIT), 64 frames at 256 px through
a single spatio-temporal encoder — into the same cache contract:

```
<DATA>/cached_features/vjepa2-vitl/<clip_id>.npy   float16  (32, 1024)
```

Nothing else changes. `train.ipynb` sets `VIDEO_ENCODER = "vjepa2-vitl"`, `model.configure`
reads the width off the cache, and the variants build a 1024-wide input layer. The audio and
text caches stay exactly where they are, so the comparison `tia` on DINOv2 vs `tia` on
V-JEPA 2 differs in one thing: the visual encoder.

| | |
|---|---|
| **Time** | ~1–2 h on a T4 (8,192 tokens per clip for a ViT-L) |
| **Writes** | `cached_features/vjepa2-vitl/` + `manifest.json` (~107 MB) |
| **Resumable** | yes — an existing `.npy` is skipped |
| **Memory** | batch 1, fp16; if it OOMs, `FRAMES = 32` (manifest records it) |

## 0 — Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
if not os.path.isdir("/content/DeepLearningCourse/.git"):
    !git clone -q https://github.com/mattsays/DeepLearningCourse /content/DeepLearningCourse
%cd /content/DeepLearningCourse/lie-detector
!git pull -q
import sys; sys.path.insert(0, '.')

import subprocess


def pip(*arguments):
    """Install into the interpreter running this kernel, and stop the cell if pip fails.

    Not `!pip`: that goes through a shell, so it can land in a different environment than the
    kernel, and its exit status is not an error — a failed install would be discovered six
    cells later. Not `-q` either: that hides the reason, which is how a resolver conflict turns
    into a bare `CalledProcessError`.
    """
    subprocess.run([sys.executable, "-m", "pip", "install", *arguments], check=True)


# V-JEPA 2 is newer than the transformers the rest of the repo pins; the exact version is
# recorded in the manifest so a cache stays identifiable after the Colab image moves.
pip("-U", "transformers>=4.56,<5")

# --no-deps, exactly how the DINOv2 cache installs it: facenet-pytorch pins torch/torchvision,
# and on a newer image letting pip resolve them either replaces the torch Colab ships with or
# fails outright. The face crop has to stay comparable with the DINOv2 cache, so: same version,
# same flags.
pip("--no-deps", "facenet-pytorch==2.6.0")

import facenet_pytorch
import transformers
from facenet_pytorch import MTCNN     # import here, not eight cells later

print("transformers", transformers.__version__,
      "| facenet-pytorch", getattr(facenet_pytorch, "__version__", "ok"))


/content/DeepLearningCourse/lie-detector
transformers 4.57.6 | facenet-pytorch ok


In [3]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import torch
from facenet_pytorch import MTCNN
from PIL import Image
from transformers import AutoModel, AutoVideoProcessor

from lib.dataset import Clips, Protocol, cache_dir, stage_media

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device", device)

torch 2.14.1+cu130 | device cuda


## 1 — Settings

`FRAMES = 64` at 256 px is the configuration V-JEPA 2 was trained on. The cache keeps the
temporal axis (32 positions after the tubelet) and averages the 256 spatial patches: 64 KB per
clip instead of 16 MB.

In [ ]:
DATA = Path("/content/drive/MyDrive/dolos")

VIDEO_CACHE = "vjepa2-vitl"
HF_ID = "facebook/vjepa2-vitl-fpc64-256"
FRAMES, CROP = 64, 256
FORCE = False
LIMIT = 0

print("video cache", cache_dir(VIDEO_CACHE, DATA))

video cache /content/drive/MyDrive/dolos/cached_features/vjepa2-vitl


## 2 — Which clips

In [5]:
folds = Protocol.load(DATA / "splits")
# Decoding an mp4 over the mount pays a round trip per file: stage the clips first.
clips = Clips.load(stage_media(DATA, parts=("clips",)))
print(clips.describe_source())

split_ids = []
for fold in range(1, folds.folds + 1):
    for part in folds.subject(fold):
        split_ids += part
split_ids += folds.test()
split_ids = list(dict.fromkeys(split_ids))

assert not clips.missing_from(split_ids), "the splits reference clips absent from dataset.csv"
ids = clips.with_media(split_ids)
print(f"protocol {len(split_ids)} clips -> {len(ids)} with media on disk")
print(clips.describe(ids))

staging media: 100%|██████████| 1453/1453 [00:59<00:00, 24.50file/s, 494 MB]

staged 1453 files (494 MB) into /content/dolos-local in 59 s on 8 thread(s); 0 already there
dataset.csv sha256:8fa198b1fe71  1678 clips  1452 with media
protocol 1452 clips -> 1452 with media on disk
{'clips': 1452, 'subjects': 161, 'deception': 53.6, 'pos_weight': 0.866, 'female': 20.5, 'majority_baseline': 53.6}


## 3 — Frames and face

The same MTCNN crop as the DINOv2 cache: box squared with a margin, cut from the original
frame, resized to 256. Keeping the pixels identical is what makes the two video caches
comparable — only the encoder differs.

In [6]:
detector = MTCNN(keep_all=False, device=device, min_face_size=40, factor=0.709,
                 post_process=False)


def sample_frames(path, count=FRAMES):
    capture = cv2.VideoCapture(str(path))
    if not capture.isOpened():
        raise RuntimeError(f"cannot open {path}")
    total = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        total = sum(1 for _ in iter(capture.grab, False))
        capture.release()
        capture = cv2.VideoCapture(str(path))
    if total <= 0:
        total = count
    targets = np.linspace(0, max(total - 1, 1), count).round().astype(int)
    wanted = set(targets.tolist())
    frames, index = [], 0
    ok, frame = capture.read()
    while ok and index <= int(targets.max()):
        if index in wanted:
            frames.append(frame)
        ok, frame = capture.read()
        index += 1
    capture.release()
    while len(frames) < count:
        frames.append(frames[-1] if frames else np.zeros((CROP, CROP, 3), np.uint8))
    return frames


def face_crop(frame, margin=1.75):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    small = cv2.resize(rgb, (640, int(640 * rgb.shape[0] / rgb.shape[1])))
    # MTCNN 2.6 returns (boxes, probs): `boxes` is (N, 4) — the largest face first when
    # keep_all=False — or None when no face was found. Not a single flat row.
    found = detector.detect(Image.fromarray(small))[0]
    rows = np.empty((0, 4)) if found is None else np.asarray(found, np.float64).reshape(-1, 4)
    box = rows[0] if len(rows) else None
    height, width = rgb.shape[:2]
    if box is None:
        side = min(height, width)
        top, left = (height - side) // 2, (width - side) // 2
        crop = rgb[top:top + side, left:left + side]
    else:
        scale = width / small.shape[1]
        x0, y0, x1, y1 = [coordinate * scale for coordinate in box]
        cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
        side = max(x1 - x0, y1 - y0) * margin
        x0, y0 = int(max(cx - side / 2, 0)), int(max(cy - side / 2, 0))
        x1, y1 = int(min(cx + side / 2, width)), int(min(cy + side / 2, height))
        crop = rgb[y0:y1, x0:x1]
    return cv2.resize(crop, (CROP, CROP), interpolation=cv2.INTER_AREA)


def video_tensor(path):
    crops = [face_crop(frame) for frame in sample_frames(path)]
    array = np.stack(crops)
    return torch.from_numpy(array).permute(0, 3, 1, 2)      # (T, 3, H, W) uint8

## 4 — The encoder

`AutoModel` + `AutoVideoProcessor`, `no_grad`, fp16. `get_vision_features` returns one vector
per token; `pool_vjepa2` averages the 256 spatial patches of each time step.

In [7]:
model = AutoModel.from_pretrained(HF_ID).to(device).eval()
processor = AutoVideoProcessor.from_pretrained(HF_ID)
for parameter in model.parameters():
    parameter.requires_grad = False
WIDTH = model.config.hidden_size
revision = getattr(model.config, "_commit_hash", None)
print(f"V-JEPA 2 | width {WIDTH} | revision {revision}")

with torch.no_grad():
    inputs = processor(video_tensor(clips[ids[0]].video), return_tensors="pt")
    inputs = {key: value.to(device) for key, value in inputs.items()}
    tokens = model.get_vision_features(**inputs)
temporal = tokens.reshape(tokens.shape[0], -1, 256, tokens.shape[-1]).mean(2)
print("raw", tuple(tokens.shape), "-> pooled", tuple(temporal.shape))
assert temporal.shape[1] == FRAMES // 2

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/785 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.30G [00:00<?, ?B/s]

video_preprocessor_config.json: 0.00B [00:00, ?B/s]

V-JEPA 2 | width 1024 | revision b3c1679b7c34d3255ef3547f27c7b226aefab26f
raw (1, 8192, 1024) -> pooled (1, 32, 1024)


## 5 — The cache

In [8]:
def settings():
    return {"encoder": VIDEO_CACHE, "hf_id": HF_ID, "revision": revision,
            "weights": "pretrained", "frames": FRAMES, "crop": CROP,
            "pool": "mean_spatial", "tokens": FRAMES // 2, "width": WIDTH,
            "dtype": "float16"}


class Cache:
    def __init__(self, directory, meta, force=False):
        self.dir = Path(directory)
        self.dir.mkdir(parents=True, exist_ok=True)
        self.manifest = self.dir / "manifest.json"
        for stale in self.dir.glob(".*.tmp.npy"):
            stale.unlink()
        previous = json.loads(self.manifest.read_text()) if self.manifest.exists() else None
        if force and previous:
            for path in self.dir.glob("*.npy"):
                path.unlink()
            previous = None
        if previous and previous.get("settings") != meta:
            raise RuntimeError(f"{self.dir.name}: settings differ — FORCE = True to rebuild")
        self.meta = previous or {"settings": meta, "count": 0, "failures": {}}
        if not previous:
            self.meta["created"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
            self.save()

    def save(self):
        self.manifest.write_text(json.dumps(self.meta, indent=2))

    def has(self, clip_id):
        return (self.dir / f"{clip_id}.npy").exists()

    def write(self, clip_id, array):
        temporary = self.dir / f".{clip_id}.tmp.npy"
        np.save(temporary, array.astype(np.float16))
        temporary.rename(self.dir / f"{clip_id}.npy")

    def pending(self, clip_ids):
        return [clip_id for clip_id in clip_ids if not self.has(clip_id)]


cache = Cache(cache_dir(VIDEO_CACHE, DATA), settings(), force=FORCE)
cache.meta["dataset_fingerprint"] = clips.fingerprint
cache.save()

## 6 — Encode

In [9]:
from tqdm import tqdm     # the console bar: it renders in Colab even when the widget does not


def encode_all(clip_ids, limit=0):
    todo = cache.pending(clip_ids)
    if limit:
        todo = todo[:limit]
    print(f"{len(todo)} clips to encode ({len(clip_ids) - len(todo)} already cached)")
    started = time.perf_counter()

    progress = tqdm(todo, desc="encoding", unit="clip", dynamic_ncols=True)
    for position, clip_id in enumerate(progress, 1):
        try:
            inputs = processor(video_tensor(clips[clip_id].video), return_tensors="pt")
            inputs = {key: value.to(device) for key, value in inputs.items()}
            with torch.no_grad(), torch.autocast(device_type=device.type,
                                                 dtype=torch.float16,
                                                 enabled=device.type == "cuda"):
                tokens = model.get_vision_features(**inputs)
            pooled = tokens.float().reshape(1, -1, 256, WIDTH).mean(2)[0]
            cache.write(clip_id, pooled.cpu().numpy())
            cache.meta["failures"].pop(clip_id, None)
        except RuntimeError as error:
            if "out of memory" in str(error):
                progress.close()
                raise SystemExit("CUDA OOM at 64 frames: set FRAMES = 32 and FORCE = True.") from error
            cache.meta["failures"][clip_id] = str(error)[:200]
            progress.write(f"  {clip_id}: {str(error)[:80]}")   # above the bar, not through it
        except Exception as error:
            cache.meta["failures"][clip_id] = str(error)[:200]
            progress.write(f"  {clip_id}: {type(error).__name__}: {str(error)[:80]}")

        rate = position / (time.perf_counter() - started)
        progress.set_postfix_str(f"{rate:.2f} clips/s, {len(cache.meta['failures'])} failed")
        if position % 25 == 0 or position == len(todo):
            cache.save()                  # the manifest is what makes an interrupted run resumable

    cache.meta["count"] = len(list(cache.dir.glob("*.npy")))
    cache.save()
    print("cached", cache.meta["count"], "arrays")


encode_all(ids[:3])


0 clips to encode (3 already cached)


encoding: 0clip [00:00, ?clip/s]

cached 1390 arrays


## 7 — Full run

In [10]:
encode_all(ids)

62 clips to encode (1390 already cached)


encoding: 100%|██████████| 62/62 [03:07<00:00,  3.03s/clip, 0.33 clips/s, 0 failed]

cached 1452 arrays


## 8 — Verify, then switch the backbone

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

directory = cache_dir(VIDEO_CACHE, DATA)
files = sorted(directory.glob("*.npy"))
size_mb = sum(path.stat().st_size for path in files) / 1e6
sample = np.load(files[0]) if files else np.zeros((0,))
absent = [clip_id for clip_id in ids if not (directory / f"{clip_id}.npy").exists()]
print(f"{VIDEO_CACHE:14} {len(files):5d} files  {size_mb:7.1f} MB  shape {sample.shape}  "
      f"missing {len(absent)}")

train_ids, val_ids, _ = folds.subject(1)
train_ids = [c for c in train_ids if (directory / f"{c}.npy").exists()]
val_ids = [c for c in val_ids if (directory / f"{c}.npy").exists()]
if train_ids and val_ids:
    x_train = np.stack([np.load(directory / f"{c}.npy").mean(0) for c in train_ids])
    x_val = np.stack([np.load(directory / f"{c}.npy").mean(0) for c in val_ids])
    probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.1))
    probe.fit(x_train, clips.labels(train_ids))
    print(f"video-only probe on fold 1 val: "
          f"AUC {roc_auc_score(clips.labels(val_ids), probe.predict_proba(x_val)[:, 1]):.3f}")
print("(plumbing check — the number that counts comes from train.ipynb)")

vjepa2-vitl     1452 files     95.3 MB  shape (32, 1024)  missing 0
video-only probe on fold 1 val: AUC 0.611
(plumbing check — the number that counts comes from train.ipynb)


## 9 — Next

In `train.ipynb`, set `VIDEO_ENCODER = "vjepa2-vitl"`. `model.configure` picks the 1024-d
width off the cache, and every variant (including `congruence`) works unchanged. Run the same
variants you ran on DINOv2 and compare on the same folds and seeds — that paired comparison is
the result.